# Sentiment Analysis - BERT Model

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
import os
import sys

src_path = os.path.abspath('../src')
sys.path.append(src_path)

from preprocessing.data_utils import DataUtils
from models import train_bert

In [ ]:
# check GPU availability
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

In [ ]:
def tokenize(df, model_name, tokenizer, max_length=128, batch_size=16):
    # remove rows where 'Review' is NaN or empty
    df = df.dropna(subset=['Review']).copy()
    df = df[df['Review'].str.strip() != '']

    # initialize tokenizer
    tokenizer = BertTokenizer.from_pretrained(model_name)

    # extract reviews and polarity
    reviews = df['Review']
    polarity = df['Polarity']
    
    # tokenize the reviews
    inputs = tokenizer(
        reviews.tolist(),
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors='tf'
    )
    
    # create a TensorFlow dataset
    return tf.data.Dataset.from_tensor_slices(({
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask']
    }, polarity)).batch(batch_size)

# 1. BERT Model - Amazon Books Reviews

## 1.1. Loading Data

In [ ]:
# load the dataset
amazon_file = '../data/preprocessed/amazon.csv'
amazon_df = pd.read_csv(amazon_file, engine='python', encoding="ISO-8859-1")

In [ ]:
# preview the data
amazon_df.head()

In [ ]:
train_val_amazon_df, test_amazon_df = train_test_split(
    amazon_df, 
    test_size=0.15, 
    random_state=42, 
    stratify=amazon_df['Polarity']
)

In [ ]:
train_amazon_df, val_amazon_df = train_test_split(
    train_val_amazon_df, 
    test_size=0.1765, # 15% of the original data
    random_state=42, 
    stratify=train_val_amazon_df['Polarity']
)

In [ ]:
DataUtils.count_reviews_by_polarity(train_amazon_df)

In [ ]:
DataUtils.count_reviews_by_polarity(val_amazon_df)

In [ ]:
DataUtils.count_reviews_by_polarity(test_amazon_df)

In [ ]:
DataUtils.plot_token_distribution(train_amazon_df, "Amazon Train Set")

In [ ]:
DataUtils.plot_token_distribution(val_amazon_df, "Amazon Validation Set")

In [ ]:
DataUtils.plot_token_distribution(test_amazon_df, "Amazon Test Set")

## 1.2. Data Preprocessing

In [ ]:
model_name = 'bert-base-uncased' # tokenizer and model name

In [ ]:
train_amazon = tokenize(
    df=train_amazon_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
val_amazon = tokenize(
    df=val_amazon_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
test_amazon = tokenize(
    df=test_amazon_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

## 1.3. Model Training

In [ ]:
# initialize model
model = TFBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
# compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
model.compile(optimizer=optimizer, loss=model.compute_loss, metrics=['accuracy'])

In [ ]:
# define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
# train the model
history = model.fit(
    train_amazon,
    validation_data=val_amazon,
    epochs=4,
    callbacks=[early_stopping]
)

In [ ]:
# plot loss on training and validation data
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
val_predictions = model.predict(val_amazon)
predicted_labels = np.argmax(val_predictions.logits, axis=1)
true_labels = np.concatenate([y.numpy() for _, y in val_amazon], axis=0)
print(classification_report(true_labels, predicted_labels))

In [ ]:
train_bert.evaluate_bert(model, test_amazon)

## 1.6. Model Evaluation on Augmented Data

In [ ]:
# load the augmented test sets
test_amazon_char_swap_file = '../data/preprocessed/amazon_test_char_swap.csv'
test_amazon_char_swap_df = pd.read_csv(test_amazon_char_swap_file, engine='python', encoding="ISO-8859-1")

test_amazon_embedding_file = '../data/preprocessed/amazon_test_embedding.csv'
test_amazon_embedding_df = pd.read_csv(test_amazon_embedding_file, engine='python', encoding="ISO-8859-1")

test_amazon_eda_file = '../data/preprocessed/amazon_test_eda.csv'
test_amazon_eda_df = pd.read_csv(test_amazon_eda_file, engine='python', encoding="ISO-8859-1")

In [ ]:
test_amazon_char_swap = tokenize(
    df=test_amazon_char_swap_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
test_amazon_embedding = tokenize(
    df=test_amazon_embedding_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
test_amazon_eda = tokenize(
    df=test_amazon_eda_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
train_bert.evaluate_bert(model, test_amazon_char_swap)

In [ ]:
train_bert.evaluate_bert(model, test_amazon_embedding)

In [ ]:
train_bert.evaluate_bert(model, test_amazon_eda)

## 1.7. Save Model

In [ ]:
model.save_pretrained("../outputs/models/bert/amazon_bert_model")

# 2. BERT Model - Sentiment140

## 2.1. Loading Data

In [ ]:
# load the dataset
sentiment140_file = '../data/preprocessed/sentiment140.csv'
sentiment140_df = pd.read_csv(sentiment140_file, engine='python', encoding="ISO-8859-1")

In [ ]:
# preview the data
sentiment140_df.head()

In [ ]:
train_val_sentiment140_df, test_sentiment140_df = train_test_split(
    sentiment140_df, 
    test_size=0.15, 
    random_state=42, 
    stratify=sentiment140_df['Polarity']
)

In [ ]:
train_sentiment140_df, val_sentiment140_df = train_test_split(
    train_val_sentiment140_df, 
    test_size=0.1765, # 15% of the original data
    random_state=42, 
    stratify=train_val_sentiment140_df['Polarity']
)

In [ ]:
DataUtils.count_reviews_by_polarity(train_sentiment140_df)

In [ ]:
DataUtils.count_reviews_by_polarity(val_sentiment140_df)

In [ ]:
DataUtils.count_reviews_by_polarity(test_sentiment140_df)

In [ ]:
DataUtils.plot_token_distribution(train_sentiment140_df, "Sentiment140 Train Set")

In [ ]:
DataUtils.plot_token_distribution(val_sentiment140_df, "Sentiment140 Validation Set")

In [ ]:
DataUtils.plot_token_distribution(test_sentiment140_df, "Sentiment140 Test Set")

## 2.2. Data Preprocessing

In [ ]:
model_name = 'bert-base-uncased' # tokenizer and model name

In [ ]:
train_sentiment140 = tokenize(
    df=train_sentiment140_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
val_sentiment140 = tokenize(
    df=val_sentiment140_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
test_sentiment140 = tokenize(
    df=test_sentiment140_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

## 2.3. Model Training

In [ ]:
# initialize model
model = TFBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
# compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
model.compile(optimizer=optimizer, loss=model.compute_loss, metrics=['accuracy'])

In [ ]:
# define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
# train the model
history = model.fit(
    train_sentiment140,
    validation_data=val_sentiment140,
    epochs=4,
    callbacks=[early_stopping]
)

In [ ]:
# plot loss on training and validation data
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
val_predictions = model.predict(val_sentiment140)
predicted_labels = np.argmax(val_predictions.logits, axis=1)
true_labels = np.concatenate([y.numpy() for _, y in val_sentiment140], axis=0)
print(classification_report(true_labels, predicted_labels))

In [ ]:
train_bert.evaluate_bert(model, test_sentiment140)

## 2.6. Model Evaluation on Augmented Data

In [ ]:
# load the augmented test sets
test_sentiment140_char_swap_file = '../data/preprocessed/sentiment140_test_char_swap.csv'
test_sentiment140_char_swap_df = pd.read_csv(test_sentiment140_char_swap_file, engine='python', encoding="ISO-8859-1")

test_sentiment140_embedding_file = '../data/preprocessed/sentiment140_test_embedding.csv'
test_sentiment140_embedding_df = pd.read_csv(test_sentiment140_embedding_file, engine='python', encoding="ISO-8859-1")

test_sentiment140_eda_file = '../data/preprocessed/sentiment140_test_eda.csv'
test_sentiment140_eda_df = pd.read_csv(test_sentiment140_eda_file, engine='python', encoding="ISO-8859-1")

In [ ]:
test_sentiment140_char_swap = tokenize(
    df=test_sentiment140_char_swap_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
test_sentiment140_embedding = tokenize(
    df=test_sentiment140_embedding_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
test_sentiment140_eda = tokenize(
    df=test_sentiment140_eda_df, 
    model_name=model_name,
    max_length=64,
    batch_size=32
)

In [ ]:
train_bert.evaluate_bert(model, test_sentiment140_char_swap)

In [ ]:
train_bert.evaluate_bert(model, test_sentiment140_embedding)

In [ ]:
train_bert.evaluate_bert(model, test_sentiment140_eda)

## 2.7. Save Model

In [ ]:
model.save_pretrained("../outputs/models/bert/sentiment140_bert_model")

# 3. BERT Model - Yelp Reviews

## 3.1. Loading Data

In [ ]:
# load the dataset
yelp_file = '../data/preprocessed/yelp.csv'
yelp_df = pd.read_csv(yelp_file, engine='python', encoding="ISO-8859-1")

In [ ]:
# preview the data
yelp_df.head()

In [ ]:
train_val_yelp_df, test_yelp_df = train_test_split(
    yelp_df, 
    test_size=0.15, 
    random_state=42, 
    stratify=yelp_df['Polarity']
)

In [ ]:
train_yelp_df, val_yelp_df = train_test_split(
    train_val_yelp_df, 
    test_size=0.1765, # 15% of the original data
    random_state=42, 
    stratify=train_val_yelp_df['Polarity']
)

In [ ]:
DataUtils.count_reviews_by_polarity(train_yelp_df)

In [ ]:
DataUtils.count_reviews_by_polarity(val_yelp_df)

In [ ]:
DataUtils.count_reviews_by_polarity(test_yelp_df)

In [ ]:
DataUtils.plot_token_distribution(train_yelp_df, "Yelp Train Set")

In [ ]:
DataUtils.plot_token_distribution(val_yelp_df, "Yelp Validation Set")

In [ ]:
DataUtils.plot_token_distribution(test_yelp_df, "Yelp Test Set")

## 3.2. Data Preprocessing

In [ ]:
model_name = 'bert-base-uncased' # tokenizer and model name
max_length = 256
batch_size = 8

In [ ]:
train_yelp = tokenize(
    df=train_yelp_df, 
    model_name=model_name,
    max_length=max_length,
    batch_size=batch_size
)

In [ ]:
val_yelp = tokenize(
    df=val_yelp_df, 
    model_name=model_name,
    max_length=max_length,
    batch_size=batch_size
)

In [ ]:
test_yelp = tokenize(
    df=test_yelp_df, 
    model_name=model_name,
    max_length=max_length,
    batch_size=batch_size
)

## 3.3. Model Training

In [ ]:
# initialize model
model = TFBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
# compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
model.compile(optimizer=optimizer, loss=model.compute_loss, metrics=['accuracy'])

In [ ]:
# define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
# train the model
history = model.fit(
    train_yelp,
    validation_data=val_yelp,
    epochs=4,
    callbacks=[early_stopping]
)

In [ ]:
# plot loss on training and validation data
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
val_predictions = model.predict(val_yelp)
predicted_labels = np.argmax(val_predictions.logits, axis=1)
true_labels = np.concatenate([y.numpy() for _, y in val_yelp], axis=0)
print(classification_report(true_labels, predicted_labels))

In [ ]:
train_bert.evaluate_bert(model, test_yelp)

## 3.6. Model Evaluation on Augmented Data

In [ ]:
# load the augmented test sets
test_yelp_char_swap_file = '../data/preprocessed/yelp_test_char_swap.csv'
test_yelp_char_swap_df = pd.read_csv(test_yelp_char_swap_file, engine='python', encoding="ISO-8859-1")

test_yelp_transform_file = '../data/preprocessed/yelp_test_transform.csv'
test_yelp_transform_df = pd.read_csv(test_yelp_transform_file, engine='python', encoding="ISO-8859-1")

test_yelp_combined_file = '../data/preprocessed/yelp_test_combined.csv'
test_yelp_combined_df = pd.read_csv(test_yelp_combined_file, engine='python', encoding="ISO-8859-1")

In [ ]:
test_yelp_char_swap = tokenize(
    df=test_yelp_char_swap_df, 
    model_name=model_name,
    max_length=max_length,
    batch_size=batch_size
)

In [ ]:
test_yelp_transform = tokenize(
    df=test_yelp_transform_df, 
    model_name=model_name,
    max_length=max_length,
    batch_size=batch_size
)

In [ ]:
test_yelp_combined = tokenize(
    df=test_yelp_combined_df, 
    model_name=model_name,
    max_length=max_length,
    batch_size=batch_size
)

In [ ]:
train_bert.evaluate_bert(model, test_yelp_char_swap)

In [ ]:
train_bert.evaluate_bert(model, test_yelp_transform)

In [ ]:
train_bert.evaluate_bert(model, test_yelp_combined)

## 3.7. Save Model

In [ ]:
model.save_pretrained("../outputs/models/bert/yelp_bert_model")